# 전처리 방법론 비교

동일한 원천 캔들에 한 번에 한 가지 전처리 결정만 바꿔 샘플 수, 클래스 분포, 중첩, 클리닝 영향을 비교한다. 이 노트북은 `pivot.dataset.build.run_preprocess`를 직접 호출하므로 Lab·batch와 동일한 결과를 사용한다. 원천 parquet는 읽기 전용이다.

## 실험 원칙

- 기준 프리셋에서 변수 하나만 변경한다.
- 샘플 수 증가만으로 우수하다고 판단하지 않는다. 클래스 균형, 90% 이상 overlap, 제외 비율을 함께 본다.
- `filter` cleaning은 세그먼트 경계를 넘는 샘플을 만들지 않는다.
- 최종 판단은 동일 데이터셋 split/seed의 validation macro F1과 클래스별 precision/recall로 내린다.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from pivot.config import PreprocessPreset, Timeframe
from pivot.dataset.build import run_preprocess
from pivot.ingestion.cache import cache_path, load_cache
from pivot.ingestion.fetch import cache_broker

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA_ROOT = ROOT / 'data'

SYMBOLS = ['005930', '000660', '005380', '006400', '105560']
TIMEFRAME = Timeframe.from_code('day')
REGION = 'domestic'
EXCHANGE = ''


In [ ]:
def load_symbol(symbol: str) -> pd.DataFrame:
    path = cache_path(
        DATA_ROOT, cache_broker(REGION, EXCHANGE), TIMEFRAME.code, symbol
    )
    frame = load_cache(path)
    if frame is None or frame.empty:
        raise FileNotFoundError(f'{symbol} {TIMEFRAME.code} 캐시가 없습니다: {path}')
    return frame

frames = {symbol: load_symbol(symbol) for symbol in SYMBOLS}
pd.DataFrame(
    [
        {
            'symbol': symbol,
            'bars': len(frame),
            'first': frame.index.min(),
            'last': frame.index.max(),
        }
        for symbol, frame in frames.items()
    ]
).set_index('symbol')

In [ ]:
BASE = PreprocessPreset(name='research-baseline', timeframe=TIMEFRAME)

def variant(name: str, **sections: dict) -> PreprocessPreset:
    payload = BASE.model_dump(mode='json')
    payload['name'] = name
    for section, values in sections.items():
        payload[section].update(values)
    return PreprocessPreset.model_validate(payload)

PRESETS = {
    'baseline': BASE,
    'tie_all': variant('tie_all', fractal={'tie_policy': 'all'}),
    'legacy_pairing': variant(
        'legacy_pairing', labeling={'sample_pairing': 'latest_opposite_v1'}
    ),
    'cls2_drop': variant('cls2_drop', labeling={'mode': 'cls2_drop'}),
    'cleaning_filter': variant('cleaning_filter', cleaning={'mode': 'filter'}),
    'ignore_none': variant('ignore_none', labeling={'ignore_rule': 'none'}),
    'min_swing_3pct': variant(
        'min_swing_3pct', labeling={'ignore_swing_pct': 3.0}
    ),
}
[(name, preset.fractal.tie_policy, preset.labeling.model_dump(), preset.cleaning.mode)
 for name, preset in PRESETS.items()]

In [ ]:
def summarize(symbol: str, name: str, result) -> dict:
    stats = result.stats
    overlap = stats['overlap_clusters']
    cleaning = stats['cleaning']
    counts = stats['class_counts']
    total = max(stats['samples'], 1)
    lengths = pd.Series([sample.length for sample in result.samples], dtype='float64')
    return {
        'symbol': symbol,
        'variant': name,
        'points': stats['points'],
        'samples': stats['samples'],
        'label_0_pct': counts[0] / total,
        'label_1_pct': counts[1] / total,
        'label_2_pct': counts[2] / total,
        'median_length': lengths.median(),
        'p95_length': lengths.quantile(0.95),
        'dropped_nan': stats['dropped_nan'],
        'plateau_removed': overlap['dropped_plateau_points'],
        'overlap_clusters': overlap['sample_clusters'],
        'redundant_samples': overlap['redundant_samples'],
        'cleaning_removed_pct': cleaning['removed_ratio'],
        'segments': cleaning['segments'],
    }

results = {}
rows = []
for symbol, frame in frames.items():
    for name, preset in PRESETS.items():
        result = run_preprocess(frame, preset)
        results[(symbol, name)] = result
        rows.append(summarize(symbol, name, result))

summary = pd.DataFrame(rows)
display(summary.head())

In [ ]:
metrics = [
    'samples', 'label_0_pct', 'label_1_pct', 'label_2_pct',
    'median_length', 'p95_length', 'dropped_nan',
    'plateau_removed', 'overlap_clusters', 'redundant_samples',
    'cleaning_removed_pct', 'segments',
]
aggregate = summary.groupby('variant')[metrics].mean().sort_index()
display(aggregate.style.format({
    'label_0_pct': '{:.1%}',
    'label_1_pct': '{:.1%}',
    'label_2_pct': '{:.1%}',
    'cleaning_removed_pct': '{:.2%}',
}))

baseline = aggregate.loc['baseline']
delta = aggregate.subtract(baseline).drop(index='baseline')
display(delta.style.format('{:+.3f}'))

In [ ]:
# 특정 종목·방법의 라벨 지점과 샘플 구간을 직접 검수한다.
INSPECT_SYMBOL = SYMBOLS[0]
INSPECT_VARIANT = 'baseline'
inspect = results[(INSPECT_SYMBOL, INSPECT_VARIANT)]

sample_rows = pd.DataFrame(
    [
        {
            'sample_index': index,
            'label': sample.label,
            'kind': sample.kind,
            'length': sample.length,
            'start': inspect.frame.index[sample.start_position],
            'end': inspect.frame.index[sample.end_position],
            'swing_pct': abs(
                inspect.frame['Close'].iloc[sample.end_position]
                / inspect.frame['Close'].iloc[sample.start_position]
                - 1.0
            ) * 100.0,
        }
        for index, sample in enumerate(inspect.samples)
    ]
)
display(sample_rows.describe(include='all'))
display(sample_rows.head(20))

## 결론 기록

아래 항목을 실험 후 채운다.

- 채택할 tie policy와 근거:
- 채택할 pairing과 근거:
- label 2 유지/제외 결정:
- cleaning `report_only` 대비 `filter`의 샘플 손실 허용 여부:
- 최소 swing threshold 후보:
- 다음 데이터셋 A/B 실험에 사용할 프리셋 이름과 seed: